In [ ]:
# BLOCO 1: Instalações que Exigem Reinício

# 1. Downgrade preventivo do Numpy (para evitar conflitos com AV)
!pip install "numpy<2.0"

# 2. Instala AV (biblioteca de decodificação) na versão estável (evita o Silent Hang do MP3)
!pip install "av==15.0.0"

print("\n-------------------------------------------------------------------------")
print("✅ INSTALAÇÕES INICIAIS CONCLUÍDAS.")
print("🔴 AÇÃO CRÍTICA: Você DEVE REINICIAR o ambiente agora para estabilizar o numpy.")
print("Vá em Runtime (Ambiente de Execução) > Restart session (Reiniciar Sessão) ou clique no botão de reinício.")
print("-------------------------------------------------------------------------")



In [ ]:
# BLOCO 2: Instalações Finais (Execute SOMENTE APÓS o REINÍCIO da Sessão do Bloco 1)

# 1. Força a reinstalação do PyTorch + TorchVision sincronizados
#    A versão 2.3.1 é compatível com o ambiente CUDA 11.8 da GPU T4 do Colab.
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu118

# 2. Instala WhisperX e dependências 
!pip install git+https://github.com/m-bain/whisperX.git --no-deps
!pip install ffmpeg-python pandas scipy "pyannote.audio==3.3.2" ctranslate2 faster-whisper

print("\n✅ INSTALAÇÃO E CONFIGURAÇÃO COMPLETAS. O ambiente está pronto.")

In [ ]:
# BLOCO 3: MONTAGEM DO DRIVE E CONFIGURAÇÃO DE SESSÃO
from google.colab import drive
drive.mount('/content/drive')

import os

# ----------------------------------------------------------------------
# EDITE APENAS ESTAS DUAS VARIÁVEIS PARA CADA NOVA SESSÃO:
# ----------------------------------------------------------------------

# O ID da sessão atual (e.g., "THoL01", "ID56", "DiT10")
ID_SESSAO = "THoL03" 

# O caminho base da sua pasta de projeto no Google Drive
PASTA_PROJETO = "/content/drive/MyDrive/rpg-transcription"

# ----------------------------------------------------------------------
# VARIÁVEIS DERIVADAS (Não precisam ser editadas)
# ----------------------------------------------------------------------

# Arquivo de entrada para a transcrição
ARQUIVO_WAV_ENTRADA = os.path.join(PASTA_PROJETO, f"{ID_SESSAO}w.wav")

# Arquivo TXT de saída da transcrição (será lido pelo Gemini)
ARQUIVO_TXT_SAIDA = os.path.join(PASTA_PROJETO, f"{ID_SESSAO}_transcricao_final.txt")

# Arquivo Markdown (Crônica) de saída do Gemini
ARQUIVO_CRONICA_SAIDA = os.path.join(PASTA_PROJETO, f"{ID_SESSAO}_cronica.md")


print("Configuração de sessão concluída:")
print(f"  ID da Sessão: {ID_SESSAO}")
print(f"  Entrada WAV: {ARQUIVO_WAV_ENTRADA}")
print(f"  Saída TXT: {ARQUIVO_TXT_SAIDA}")
print(f"  Saída Crônica: {ARQUIVO_CRONICA_SAIDA}")

# ----------------------------------------------------------------------
# VARIÁVEIS DE CONTEÚDO (Podem ser editadas conforme mais conteúdo precisar ser adicionado)
# ----------------------------------------------------------------------

# Glossário para ajudar o Whisper a entender nomes próprios e termos das sessões
NOMES_DA_CAMPANHA = [
# Termos de sistema
"D20",
"Iniciativa",
"Perception",
"Insight",
"Investigation",
# Nomes de personagens de jogadores - Zéfiros
"Pharah",
"Janos",
"Bjorn",
"Bbo",
"Akros",
# Nomes de personagens de jogadores - Inferno
"Kay",
"Sarfan Thuranni",
"Vander Bremen",
"Purcival Purcy Bartolomeow",
"Penélope",
"Orog",
# Nomes de outros personagens
"De Lata",
"Pedrinha",
"Alustriel",
"Mordenkainen",
"Tasha",
"Zéfiros",
"Tiamat",
"Zariel",
"Vecna",
# Nomes de locais
"Eberron",
"Elturel",
"Elturgard",
"Baldur’s Gate",
"Candlekeep",
"Icewind Dale",
"Baróvia",
"Faerûn"
]

GLOSSARIO_NOMES = ", ".join(NOMES_DA_CAMPANHA)


In [ ]:
# BLOCO 4: Execução da Transcrição (Faster-Whisper)
from faster_whisper import WhisperModel
print("Iniciando a transcrição com Faster-Whisper.")

# 1a. Se estiver usando GPU: Carregar o modelo (usa a GPU e o motor ctranslate2)
model = WhisperModel("large-v2", device="cuda", compute_type="float16")

# 1b. Se estiver usando CPU: Carregar o modelo (usa a CPU e o motor ctranslate2)
# model = WhisperModel("large-v2", device="cpu", compute_type="int8")

# 2. Transcrição
segments, info = model.transcribe(
    ARQUIVO_WAV_ENTRADA,
    language="pt", 
    vad_filter=True, 
    word_timestamps=False, 
    initial_prompt=GLOSSARIO_NOMES 
)

# 3. Processar e salvar o texto
with open(ARQUIVO_TXT_SAIDA, "w", encoding="utf-8") as f:
    print(f"Salvando o texto em: {ARQUIVO_TXT_SAIDA}")
    
    # Escreve cada segmento como uma linha no arquivo
    for segment in segments:
        f.write(f"{segment.text}\n")
        
print("\nTranscrição CONCLUÍDA e salva no Google Drive!")



In [ ]:
# BLOCO 5: Geração da Crônica com Gemini API
import os
import requests
from google import genai
from google.genai.errors import APIError
from google.colab import userdata # Importa o módulo para acessar Secrets

# --- CHAVE DE API: NOVO MÉTODO DE ACESSO ---
# 1. Tenta acessar o Secret. A variável GEMINI_API_KEY será definida como o valor
#    do Secret, ou será None se o acesso falhar ou a chave não existir.

# O NOME DO SECRET DEVE SER EXATAMENTE 'GEMINI_API_KEY'
GEMINI_API_KEY = None # Inicializa como None por segurança

try:
    # Acessa o valor do Secret do Colab
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except KeyError:
    # Se o Secret não for encontrado, a variável permanece None e a mensagem é impressa.
    print("ERRO DE CONFIGURAÇÃO: O Secret 'GEMINI_API_KEY' não foi encontrado. Verifique se o nome está correto e se o acesso está ativado.")
# ------------------------------------------------------------

CAMINHO_TRANSCRICAO = ARQUIVO_TXT_SAIDA
CAMINHO_SAIDA_CRONICA = ARQUIVO_CRONICA_SAIDA

def gerar_cronica_gemini_automatizada():
    """Lê o arquivo de transcrição, gera a crônica e salva o resultado."""
   
    # 1. Baixar o prompt do GitHub
    url_prompt = "https://raw.githubusercontent.com/ThaisMosken/rpg_transcription/refs/heads/main/prompts/template_cronica_v1.md"
    try:
        response = requests.get(url_prompt)
        response.raise_for_status()
        prompt_base = response.text
        print("✅ Prompt base baixado do GitHub com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao baixar o prompt: {e}")
        return

    # 2. Configura o cliente da API
    try:
        # Se a chave não for válida, o cliente pode falhar aqui
        client = genai.Client(api_key=GEMINI_API_KEY)
    except NameError:
        print("ERRO: A variável GEMINI_API_KEY não está definida. Cole sua chave de API antes de prosseguir.")
        return
    except Exception as e:
        print(f"ERRO: Falha ao configurar o cliente. Verifique se a chave de API é válida. Detalhes: {e}")
        return

    # 3. Ler o conteúdo do arquivo TXT do Drive
    try:
        with open(CAMINHO_TRANSCRICAO, 'r', encoding='utf-8') as f:
            transcricao_texto = f.read()
        print(f"Transcrição lida com sucesso. Tamanho: {len(transcricao_texto)} caracteres.")
    except FileNotFoundError:
        print(f"ERRO: Arquivo de transcrição não encontrado em {CAMINHO_TRANSCRICAO}. Verifique o caminho.")
        return

    # 4. Montar o Prompt com as instruções de Crônica

    referencia_lore = "\n".join([f"- {nome}" for nome in NOMES_DA_CAMPANHA])

    prompt_completo = f"""

    **BLOCO DE TEXTO TRANSCRITO:**
    --- INÍCIO DA TRANSCRIÇÃO ---
    {transcricao_texto}
    --- FIM DA TRANSCRIÇÃO —

    **IMPORTANTE:** Use o glossário a seguir para corrigir nomes que o transcritor possa ter entendido errado: {referencia_lore}

    {prompt_base}

    """

    # 5. Chamar a API do Gemini
    print("\nEnviando solicitação ao modelo Gemini 2.5 Flash (Isso pode levar alguns segundos)...")
    generation_config = {
        "temperature": 0.7,
        "top_p": 0.95,
        "max_output_tokens": 8192,
    }

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt_completo,
            config=generation_config
        )
       
        cronica_final = response.text
       
        # 6. Verificação de Bloqueio (Safety Filter)
        if cronica_final is None:
            feedback = response.prompt_feedback
            if feedback and feedback.block_reason:
                print(f"\n❌ ERRO DE GERAÇÃO: O modelo não conseguiu gerar texto.")
                print(f"Motivo do Bloqueio (Safety Filter): {feedback.block_reason.name}")
            else:
                 print("\n⚠️ ERRO DE GERAÇÃO: O modelo não conseguiu gerar texto. (Motivo desconhecido ou timeout)")
            return

        # 7. Salvar a Crônica final no Drive
        with open(CAMINHO_SAIDA_CRONICA, 'w', encoding='utf-8') as f:
            f.write(cronica_final)
           
        print(f"\n✅ SUCESSO! Crônica gerada e salva no Drive em: {CAMINHO_SAIDA_CRONICA}")
       
    except APIError as e:
        print(f"ERRO DE API: Falha na chamada ao Gemini. Detalhes: {e}")
    except Exception as e:
        print(f"Ocorreu um erro inesperado: {e}")

# 8. Executar a função principal
gerar_cronica_gemini_automatizada()

